# Books Catalog Analysis with Python, Pandas and Web Scraping

This project collects book data from a scraping practice website and analyzes the catalog using Python and Pandas.

The goal is to demonstrate a complete data analysis workflow: web scraping, data cleaning, exploratory analysis, visualization and business interpretation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Style global des graphiques
plt.rcParams["font.family"] = ["Aptos Narrow", "Arial", "DejaVu Sans"]
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "#D9D9D9"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.color"] = "#E6E6E6"
plt.rcParams["grid.linestyle"] = "-"
plt.rcParams["grid.linewidth"] = 0.7

In [ ]:
# Palette graphique du projet
CHART_COLORS = [
    "#2FA4A9",
    "#2FA4A9BF",
    "#2FA4A980",
    "#2F5D8C",
    "#2F5D8CBF",
    "#2F5D8C80"
]

def clean_chart(
    title=None,
    xlabel=None,
    ylabel=None,
    add_labels=False,
    label_format="{:.2f}",
    label_prefix="",
    label_suffix="",
    orientation="vertical",
    label_padding=0.3,
    ylim_padding=5,
    xlim_padding=5
):
    ax = plt.gca()

    # Titres et labels
    plt.title(title, fontweight="bold", pad=15)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)

    # Supprimer les bordures inutiles
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Supprimer les traits/grilles au milieu
    ax.grid(False)

    # Ajouter les valeurs sur les barres
    if add_labels:
        for bar in ax.patches:
            if orientation == "vertical":
                value = bar.get_height()

                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    value + label_padding,
                    f"{label_prefix}{label_format.format(value)}{label_suffix}",
                    ha="center",
                    va="bottom",
                    fontsize=10
                )

            elif orientation == "horizontal":
                value = bar.get_width()

                ax.text(
                    value + label_padding,
                    bar.get_y() + bar.get_height() / 2,
                    f"{label_prefix}{label_format.format(value)}{label_suffix}",
                    ha="left",
                    va="center",
                    fontsize=10
                )

    # Ajouter de l'espace pour éviter que les labels soient coupés
    if add_labels and orientation == "vertical":
        current_ylim = ax.get_ylim()
        ax.set_ylim(current_ylim[0], current_ylim[1] + ylim_padding)

    if add_labels and orientation == "horizontal":
        current_xlim = ax.get_xlim()
        ax.set_xlim(current_xlim[0], current_xlim[1] + xlim_padding)

    plt.tight_layout()

In [ ]:
#Lecture du fichier brute
df = pd.read_csv("C:/Users/matma/OneDrive/Documents/Portefolio/Projet Python_Scrap/books_raw.csv")
df.head()
df.shape
df.info

In [ ]:
#Création du fichier
books = df.copy()

books["price"] = (
    books["price_raw"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .str.extract(r"(\d+\.\d+)")[0]
    .astype(float)
)

books["stock_available"] = pd.to_numeric(
    books["stock_available"],
    errors="coerce"
)

books["is_available"] = books["stock_available"] > 0

books_cleaned = books[
    [
        "title",
        "price",
        "availability",
        "stock_available",
        "is_available",
        "rating_text",
        "rating",
        "detail_url"
    ]
]

books_cleaned.head()

In [ ]:
total_books = len(books_cleaned)
average_price = books_cleaned["price"].mean()
median_price = books_cleaned["price"].median()
average_rating = books_cleaned["rating"].mean()
available_rate = books_cleaned["is_available"].mean() * 100

summary = pd.DataFrame({
    "Metric": ["Total books", "Average price", "Median price", "Average rating", "Availability rate"],
    "Value": [total_books, round(average_price, 2), round(median_price, 2), round(average_rating, 2), round(available_rate, 2)]
})

summary

In [ ]:
plt.figure(figsize=(8, 5))

books_cleaned["price"].plot(
    kind="hist",
    bins=30,
    color=COLOR_BLUE,
    edgecolor="white"
)

clean_chart(
    title="Distribution des prix des livres",
    xlabel="Prix (£)",
    ylabel="Nombre de livres"
)

plt.savefig("C:/Users/matma/OneDrive/Documents/Portefolio/Projet Python_Scrap/screenshots/price_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
rating_distribution = books_cleaned["rating"].value_counts().sort_index()

plt.figure(figsize=(7, 5))

plt.bar(
    rating_distribution.index,
    rating_distribution.values,
    color=CHART_COLORS[:len(rating_distribution)],
    edgecolor=CHART_COLORS[:len(rating_distribution)]
)

clean_chart(
    title="Répartition des notes",
    xlabel="Note",
    ylabel="Nombre de livres",
    add_labels=True,
    label_format="{:.0f}",
    orientation="vertical"
)

plt.xticks(rating_distribution.index)
plt.savefig("C:/Users/matma/OneDrive/Documents/Portefolio/Projet Python_Scrap/screenshots/rating_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

books_cleaned["stock_available"].plot(
    kind="hist",
    bins=20,
    color="#2FA4A9",
    edgecolor="#2FA4A9"
)

clean_chart(
    title="Distribution du stock disponible",
    xlabel="Nombre d'exemplaires disponibles",
    ylabel="Nombre de livres"
)

plt.savefig("C:/Users/matma/OneDrive/Documents/Portefolio/Projet Python_Scrap/screenshots/stock_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
top_expensive_books = books_cleaned.sort_values("price", ascending=False).head(10)

plt.figure(figsize=(10, 6))

plt.barh(
    top_expensive_books["title"],
    top_expensive_books["price"],
    color="#2FA4A9",
    edgecolor="#2FA4A9"
)

plt.gca().invert_yaxis()

clean_chart(
    title="Top 10 des livres les plus chers",
    xlabel="Prix (£)",
    add_labels=True,
    label_format="{:.2f}",
    label_prefix="£",
    orientation="horizontal",
    label_padding=0.2,
    xlim_padding=4
)

plt.xlim(0, top_expensive_books["price"].max() + 5)

plt.savefig("C:/Users/matma/OneDrive/Documents/Portefolio/Projet Python_Scrap/screenshots/top_expensive_books.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
price_by_rating = books_cleaned.groupby("rating")["price"].mean().reset_index()

plt.figure(figsize=(7, 5))

plt.bar(
    price_by_rating["rating"],
    price_by_rating["price"],
    color=CHART_COLORS[:len(price_by_rating)],
    edgecolor=CHART_COLORS[:len(price_by_rating)]
)

clean_chart(
    title="Prix moyen par note",
    xlabel="Note",
    ylabel="Prix moyen (£)",
    add_labels=True,
    label_format="{:.2f}",
    label_prefix="£",
    orientation="vertical"
)

plt.xticks(price_by_rating["rating"])
plt.savefig("C:/Users/matma/OneDrive/Documents/Portefolio/Projet Python_Scrap/screenshots/price_by_rating.png", dpi=300, bbox_inches="tight")
plt.show()

## Key Insights

- The dataset contains 1,000 books scraped from a demo online catalog.
- Most books are available in stock.
- Prices are distributed across a wide range, allowing analysis of low and high-priced items.
- Ratings are spread from 1 to 5 stars.
- The most expensive books are not necessarily the highest-rated ones.
- Web scraping was used to collect the raw data before cleaning and analysis with Pandas.